In [14]:
# IMPORTS, CONSTANTS
import os
import json
import sys
import requests
import browser_cookie3
from datetime import datetime
from zoneinfo import ZoneInfo
from dotenv import load_dotenv
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from cbb.lib import paths

load_dotenv()

LEAGUE_ID = 1039832288
SEASON = 2026

URL = (
    f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/wfba/"
    f"seasons/{SEASON}/segments/0/leagues/{LEAGUE_ID}"
)

PARAMS = [
    ("view", "mTeam"),
    ("view", "mRoster"),
    ("view", "mSettings"),
    ("view", "mMatchup"),
    ("view", "mScoreboard"),
]

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json,text/plain,*/*",
    "Referer": f"https://fantasy.espn.com/womens-basketball/league?leagueId={LEAGUE_ID}",
    "Origin": "https://fantasy.espn.com",
}

In [15]:
# FUNCTION DEFS
def get_espn_cookies():
    """
    First try .env cookies.
    If missing/expired, pull fresh cookies from your logged-in browser.
    """
    espn_s2 = os.getenv("ESPN_S2")
    swid = os.getenv("SWID")

    if espn_s2 and swid:
        return {"espn_s2": espn_s2, "SWID": swid}

    cj = browser_cookie3.chrome(domain_name=".espn.com")
    print(cj)
    cookies = {c.name: c.value for c in cj}

    if "espn_s2" not in cookies or "SWID" not in cookies:
        raise RuntimeError(
            "Could not find ESPN cookies. Log into ESPN Fantasy in Chrome first."
        )

    return {
        "espn_s2": cookies["espn_s2"],
        "SWID": cookies["SWID"],
    }


def fetch_league_data():
    session = requests.Session()
    session.cookies.update(get_espn_cookies())

    r = session.get(
        URL,
        params=PARAMS,
        headers=HEADERS,
        allow_redirects=False,
        timeout=20,
    )

    print("status:", r.status_code)
    print("content-type:", r.headers.get("content-type"))
    print("location:", r.headers.get("location"))

    if r.status_code in (301, 302, 303, 307, 308):
        raise RuntimeError("Redirected by ESPN. Cookies are invalid/expired.")

    if "application/json" not in r.headers.get("content-type", ""):
        raise RuntimeError(f"ESPN did not return JSON:\n{r.text[:500]}")


In [16]:
data = fetch_league_data()
print(data)
ET = ZoneInfo("America/New_York")

pulled_at = datetime.now(ET)

output = {
    "metadata": {
        "pulled_at": pulled_at.isoformat(),
        "pulled_at_readable": pulled_at.strftime("%Y-%m-%d %I:%M:%S %p %Z"),
        "league_id": LEAGUE_ID,
        "season": SEASON,
    },
    "data": data,
}

out_path = paths.DATA / "wnba.json"
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved fresh data pull: {output['metadata']['pulled_at_readable']}")

status: 200
content-type: application/json;charset=utf-8
location: None
None
Saved fresh data pull: 2026-05-22 02:51:17 PM EDT
